# ABS SA4 Silver Processing

This notebook processes the ABS SA4 Bronze spatial dataset into a validated Silver-layer dataset for the NSW EV data pipeline.

## Objectives

In this notebook, we will:

1. Load the ABS SA4 Bronze shapefile with GeoPandas.
2. Inspect the dataset schema, CRS, and geometry types.
3. Check for missing, empty, and invalid geometries.
4. Identify and filter NSW SA4 regions.
5. Prepare a clean ABS SA4 Silver dataset.
6. Validate the Silver output before it is used for the EV–SA4 spatial join.

## Pipeline Stage

```text
ABS SA4 Bronze
      ↓
GeoPandas Load
      ↓
Schema / CRS / Geometry Inspection
      ↓
NSW SA4 Filtering
      ↓
Geometry Validation
      ↓
ABS SA4 Silver
      ↓
EV–SA4 Spatial Join

## Step 1 — Environment and Project Check

Before reading the spatial dataset, confirm that the notebook is using the project Python environment and that GeoPandas is available.

We also inspect the current working directory because all dataset paths will be constructed relative to the project root.

In [1]:
from pathlib import Path
import geopandas as gpd

print("GeoPandas:", gpd.__version__)
print("Working directory:", Path.cwd())

GeoPandas: 1.1.4
Working directory: D:\mycode\nsw-ev-data-pipeline\notebooks


## Step 2 — Define Project Paths


The notebook is running from the `notebooks/` directory, so the project root is its parent directory.

All Bronze and Silver data paths will be constructed relative to the project root rather than using machine-specific absolute paths.

In [2]:
PROJECT_ROOT = Path.cwd().parent

ABS_BRONZE_DIR = (
    PROJECT_ROOT
    / "data"
    / "bronze"
    / "abs"
    / "SA4_2026_AUST_GDA2020"
)

ABS_SHP = ABS_BRONZE_DIR / "SA4_2026_AUST_GDA2020.shp"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ABS shapefile:", ABS_SHP)
print("Exists:", ABS_SHP.exists())

PROJECT_ROOT: D:\mycode\nsw-ev-data-pipeline
ABS shapefile: D:\mycode\nsw-ev-data-pipeline\data\bronze\abs\SA4_2026_AUST_GDA2020\SA4_2026_AUST_GDA2020.shp
Exists: True


## Step 3 — Load the ABS SA4 Shapefile


GeoPandas is used to read the ABS SA4 shapefile into a `GeoDataFrame`.

A GeoDataFrame extends a normal pandas DataFrame by adding spatial information, especially:

- a `geometry` column containing geographic features;
- a Coordinate Reference System (CRS);
- spatial operations such as filtering, intersection, and spatial joins.

At this stage, the dataset is only loaded and inspected. No transformations are applied yet.

In [3]:
sa4 = gpd.read_file(ABS_SHP)

print("Object type:", type(sa4))
print("Shape:", sa4.shape)

Object type: <class 'geopandas.geodataframe.GeoDataFrame'>
Shape: (108, 12)


## Step 4 — Inspect the Spatial Dataset


Before filtering or modifying the data, inspect its structure and spatial metadata.

We want to determine:

- how many SA4 features are present;
- which attribute columns are available;
- which fields identify SA4 regions and states;
- which CRS is used;
- which geometry types are present.

In [4]:
print("Shape:", sa4.shape)
print("\nColumns:")
print(sa4.columns.tolist())

print("\nCRS:")
print(sa4.crs)

print("\nGeometry types:")
print(sa4.geom_type.value_counts())

Shape: (108, 12)

Columns:
['SA4_CODE26', 'SA4_NAME26', 'CHG_FLAG26', 'CHG_LBL26', 'GCC_CODE26', 'GCC_NAME26', 'STE_CODE26', 'STE_NAME26', 'AUS_CODE26', 'AUS_NAME26', 'AREASQKM26', 'geometry']

CRS:
EPSG:7844

Geometry types:
MultiPolygon    48
Polygon         41
Name: count, dtype: int64


In [5]:
sa4.head()

,SA4_CODE26,SA4_NAME26,CHG_FLAG26,CHG_LBL26,GCC_CODE26,GCC_NAME26,STE_CODE26,STE_NAME26,AUS_CODE26,AUS_NAME26,AREASQKM26,geometry
0,101,Capital Region,0,No change,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,51896.2445,"MULTIPOLYGON (((150.08176 -36.377, 150.08159 -..."
1,102,Central Coast,0,No change,1GSYD,Greater Sydney,1,New South Wales,AUS,Australia,1681.0090,"MULTIPOLYGON (((151.44404 -33.49089, 151.44377..."
2,103,Central West,0,No change,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,70297.0605,"POLYGON ((150.17101 -33.78914, 150.17099 -33.7..."
3,104,Coffs Harbour - Grafton,0,No change,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,13229.7573,"MULTIPOLYGON (((153.3615 -29.35637, 153.36139 ..."
4,105,Far West and Orana,0,No change,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,339355.6495,"POLYGON ((149.21457 -32.82352, 149.21438 -32.8..."


## Step 5 — Inspect State Codes and Geometry Quality


The SA4 dataset contains state identifiers (`STE_CODE26` and `STE_NAME26`), which can be used to select New South Wales regions.

Before filtering the dataset, we inspect:

- the available state codes and names;
- the data type of the state code;
- missing or empty geometries;
- geometry validity.

This avoids assuming how the state code is stored and confirms that the spatial features are suitable for further processing.


In [6]:
print(sa4[["STE_CODE26", "STE_NAME26"]].drop_duplicates().sort_values("STE_CODE26"))

print("\nSTE_CODE26 dtype:")
print(sa4["STE_CODE26"].dtype)

print("\nMissing geometries:")
print(sa4.geometry.isna().sum())

print("\nEmpty geometries:")
print(sa4.geometry.is_empty.sum())

print("\nGeometry validity:")
print(sa4.geometry.is_valid.value_counts())

    STE_CODE26                    STE_NAME26
0            1               New South Wales
30           2                      Victoria
49           3                    Queensland
70           4               South Australia
79           5             Western Australia
91           6                      Tasmania
97           7            Northern Territory
101          8  Australian Capital Territory
104          9             Other Territories
107          Z             Outside Australia

STE_CODE26 dtype:
str

Missing geometries:
19

Empty geometries:
0

Geometry validity:
True     89
False    19
Name: count, dtype: int64


## Step 6 — Investigate Missing Geometries

The dataset contains 19 records with missing geometry.

Because the number of invalid geometries is also 19, these `False` validity results may simply correspond to records where geometry is missing rather than malformed spatial features.

Before applying any geometry repair, inspect which SA4 and state records have missing geometry and determine whether they are relevant to New South Wales.

In [7]:
missing_geometry = sa4[sa4.geometry.isna()]

print("Rows with missing geometry:", len(missing_geometry))

missing_geometry[
    ["SA4_CODE26", "SA4_NAME26", "STE_CODE26", "STE_NAME26"]
]

Rows with missing geometry: 19


,SA4_CODE26,SA4_NAME26,STE_CODE26,STE_NAME26
28,197,Migratory - Offshore - Shipping (NSW),1,New South Wales
29,199,No usual address (NSW),1,New South Wales
47,297,Migratory - Offshore - Shipping (Vic),2,Victoria
48,299,No usual address (Vic),2,Victoria
68,397,Migratory - Offshore - Shipping (Qld),3,Queensland
69,399,No usual address (Qld),3,Queensland
77,497,Migratory - Offshore - Shipping (SA),4,South Australia
78,499,No usual address (SA),4,South Australia
89,597,Migratory - Offshore - Shipping (WA),5,Western Australia
90,599,No usual address (WA),5,Western Australia


In [8]:
missing_geometry[
    ["STE_CODE26", "STE_NAME26"]
].value_counts()

STE_CODE26  STE_NAME26                  
1           New South Wales                 2
2           Victoria                        2
3           Queensland                      2
4           South Australia                 2
5           Western Australia               2
6           Tasmania                        2
7           Northern Territory              2
8           Australian Capital Territory    2
9           Other Territories               2
Z           Outside Australia               1
Name: count, dtype: int64

In [9]:
non_missing_geometry = sa4[sa4.geometry.notna()]

print(
    non_missing_geometry.geometry.is_valid.value_counts()
)

True    89
Name: count, dtype: int64


## Step 7 — Filter New South Wales SA4 Records

New South Wales is identified by `STE_CODE26 = "1"`.

The NSW records include both:

- regular SA4 regions with spatial geometries; and
- special non-spatial categories such as `Migratory - Offshore - Shipping` and `No usual address`.

The full NSW subset is retained first so that no source records are silently discarded.

A separate spatial subset will then be created for later EV–SA4 spatial joins.

In [10]:
nsw_sa4 = sa4[sa4["STE_CODE26"] == "1"].copy()

print("NSW SA4 records:", len(nsw_sa4))
print("NSW records with geometry:", nsw_sa4.geometry.notna().sum())
print("NSW records without geometry:", nsw_sa4.geometry.isna().sum())

NSW SA4 records: 30
NSW records with geometry: 28
NSW records without geometry: 2


In [11]:
nsw_sa4[
    ["SA4_CODE26", "SA4_NAME26", "STE_CODE26", "STE_NAME26"]
]

,SA4_CODE26,SA4_NAME26,STE_CODE26,STE_NAME26
0,101,Capital Region,1,New South Wales
1,102,Central Coast,1,New South Wales
2,103,Central West,1,New South Wales
3,104,Coffs Harbour - Grafton,1,New South Wales
4,105,Far West and Orana,1,New South Wales
5,106,Hunter Valley exc Newcastle,1,New South Wales
6,107,Illawarra,1,New South Wales
7,108,Mid North Coast,1,New South Wales
8,109,Murray,1,New South Wales
9,110,New England and North West,1,New South Wales


In [12]:
nsw_sa4_spatial = nsw_sa4[nsw_sa4.geometry.notna()].copy()

print("Spatial NSW SA4 records:", len(nsw_sa4_spatial))
print(nsw_sa4_spatial.geometry.is_valid.value_counts())

Spatial NSW SA4 records: 28
True    28
Name: count, dtype: int64


## Step 8 — Inspect Candidate Silver Fields

The NSW spatial subset contains 28 valid SA4 boundary records.

Before defining the Silver schema, inspect the available attributes, data types, and missing values.

The Silver layer should retain fields that are useful for:

- identifying SA4 regions;
- supporting the EV–SA4 spatial join;
- preserving useful geographic context;
- later loading the data into DuckDB Spatial.

Columns should not be removed until their downstream purpose is understood.

In [13]:
print("Columns:")
print(nsw_sa4_spatial.columns.tolist())

print("\nData types:")
print(nsw_sa4_spatial.dtypes)

print("\nMissing values:")
print(nsw_sa4_spatial.isna().sum())

Columns:
['SA4_CODE26', 'SA4_NAME26', 'CHG_FLAG26', 'CHG_LBL26', 'GCC_CODE26', 'GCC_NAME26', 'STE_CODE26', 'STE_NAME26', 'AUS_CODE26', 'AUS_NAME26', 'AREASQKM26', 'geometry']

Data types:
SA4_CODE26         str
SA4_NAME26         str
CHG_FLAG26         str
CHG_LBL26          str
GCC_CODE26         str
GCC_NAME26         str
STE_CODE26         str
STE_NAME26         str
AUS_CODE26         str
AUS_NAME26         str
AREASQKM26     float64
geometry      geometry
dtype: object

Missing values:
SA4_CODE26    0
SA4_NAME26    0
CHG_FLAG26    0
CHG_LBL26     0
GCC_CODE26    0
GCC_NAME26    0
STE_CODE26    0
STE_NAME26    0
AUS_CODE26    0
AUS_NAME26    0
AREASQKM26    0
geometry      0
dtype: int64


In [14]:
nsw_sa4_spatial.head()

,SA4_CODE26,SA4_NAME26,CHG_FLAG26,CHG_LBL26,GCC_CODE26,GCC_NAME26,STE_CODE26,STE_NAME26,AUS_CODE26,AUS_NAME26,AREASQKM26,geometry
0,101,Capital Region,0,No change,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,51896.2445,"MULTIPOLYGON (((150.08176 -36.377, 150.08159 -..."
1,102,Central Coast,0,No change,1GSYD,Greater Sydney,1,New South Wales,AUS,Australia,1681.0090,"MULTIPOLYGON (((151.44404 -33.49089, 151.44377..."
2,103,Central West,0,No change,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,70297.0605,"POLYGON ((150.17101 -33.78914, 150.17099 -33.7..."
3,104,Coffs Harbour - Grafton,0,No change,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,13229.7573,"MULTIPOLYGON (((153.3615 -29.35637, 153.36139 ..."
4,105,Far West and Orana,0,No change,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,339355.6495,"POLYGON ((149.21457 -32.82352, 149.21438 -32.8..."


## Step 9 — Evaluate Silver Schema Candidates

The NSW spatial subset is complete and contains no missing values.

Before selecting the Silver-layer columns, inspect the number of unique values and the semantic role of each attribute.

This helps distinguish between:

- identifiers required for spatial integration;
- useful regional context;
- source metadata;
- attributes that are constant or redundant in an NSW-only dataset.

In [15]:
candidate_columns = [
    "SA4_CODE26",
    "SA4_NAME26",
    "CHG_FLAG26",
    "CHG_LBL26",
    "GCC_CODE26",
    "GCC_NAME26",
    "STE_CODE26",
    "STE_NAME26",
    "AUS_CODE26",
    "AUS_NAME26",
]

for column in candidate_columns:
    print(
        f"{column}: "
        f"{nsw_sa4_spatial[column].nunique()} unique"
    )

SA4_CODE26: 28 unique
SA4_NAME26: 28 unique
CHG_FLAG26: 1 unique
CHG_LBL26: 1 unique
GCC_CODE26: 2 unique
GCC_NAME26: 2 unique
STE_CODE26: 1 unique
STE_NAME26: 1 unique
AUS_CODE26: 1 unique
AUS_NAME26: 1 unique


In [16]:
for column in [
    "CHG_FLAG26",
    "CHG_LBL26",
    "GCC_CODE26",
    "GCC_NAME26",
    "STE_CODE26",
    "STE_NAME26",
    "AUS_CODE26",
    "AUS_NAME26",
]:
    print(f"\n{column}:")
    print(nsw_sa4_spatial[column].value_counts())


CHG_FLAG26:
CHG_FLAG26
0    28
Name: count, dtype: int64

CHG_LBL26:
CHG_LBL26
No change    28
Name: count, dtype: int64

GCC_CODE26:
GCC_CODE26
1GSYD    15
1RNSW    13
Name: count, dtype: int64

GCC_NAME26:
GCC_NAME26
Greater Sydney    15
Rest of NSW       13
Name: count, dtype: int64

STE_CODE26:
STE_CODE26
1    28
Name: count, dtype: int64

STE_NAME26:
STE_NAME26
New South Wales    28
Name: count, dtype: int64

AUS_CODE26:
AUS_CODE26
AUS    28
Name: count, dtype: int64

AUS_NAME26:
AUS_NAME26
Australia    28
Name: count, dtype: int64


## Step 10 — Define the ABS Silver Dataset



The NSW spatial subset has 28 valid SA4 regions and no missing values.

The Silver layer should preserve attributes that are useful for:

- identifying SA4 regions;
- understanding regional context;
- supporting later analysis;
- performing spatial joins;
- loading spatial data into DuckDB.

Constant source metadata that does not add useful downstream information can be excluded from the Silver representation.

The original Bronze dataset remains unchanged, so removing fields from the Silver layer does not remove source lineage.

In [17]:
print(nsw_sa4_spatial["AREASQKM26"].describe())

print(
    "Non-positive areas:",
    (nsw_sa4_spatial["AREASQKM26"] <= 0).sum()
)

count        28.000000
mean      28599.918118
std       67689.885122
min          57.734100
25%         250.876375
50%        1610.124900
75%       19511.447550
max      339355.649500
Name: AREASQKM26, dtype: float64
Non-positive areas: 0


## Step 11 — Define the ABS Silver Schema


The NSW spatial subset contains 28 valid SA4 regions with complete geometry and area information.

The Silver layer will retain attributes that support:

- SA4 identification;
- Greater Sydney / Rest of NSW classification;
- regional context;
- future density and coverage analysis;
- spatial joins and spatial database loading.

Source-only attributes that are constant across all NSW spatial records and provide little downstream analytical value can be excluded.

The Bronze dataset remains unchanged and preserves the original ABS attributes.

In [20]:
silver_columns = [
    # SA4 identifier
    "SA4_CODE26",
    "SA4_NAME26",
    # GCC classification
    # 填两列
    "GCC_CODE26",
    "GCC_NAME26",
    # State context
    # 填两列
    "STE_CODE26",
    "STE_NAME26",
    # Area
    # 填一列
    "AREASQKM26",
    # Spatial geometry
    # 填一列
    "geometry",
]

In [21]:
abs_sa4_silver = nsw_sa4_spatial[silver_columns].copy()

## Step 12 — Create and Validate the ABS Silver GeoDataFrame


The selected Silver fields are extracted from the validated NSW spatial subset.

The resulting GeoDataFrame should contain:

- 28 NSW SA4 spatial regions;
- 8 selected Silver attributes;
- no missing values;
- valid geometries;
- the original spatial CRS.

In [22]:
print("Type:", type(abs_sa4_silver))
print("Shape:", abs_sa4_silver.shape)
print("Columns:", abs_sa4_silver.columns.tolist())
print("CRS:", abs_sa4_silver.crs)
print("Missing values:", abs_sa4_silver.isna().sum().sum())

Type: <class 'geopandas.geodataframe.GeoDataFrame'>
Shape: (28, 8)
Columns: ['SA4_CODE26', 'SA4_NAME26', 'GCC_CODE26', 'GCC_NAME26', 'STE_CODE26', 'STE_NAME26', 'AREASQKM26', 'geometry']
CRS: EPSG:7844
Missing values: 0


In [23]:
print("\nGeometry validity:")
print(abs_sa4_silver.geometry.is_valid.value_counts())

print("\nGeometry types:")
print(abs_sa4_silver.geom_type.value_counts())


Geometry validity:
True    28
Name: count, dtype: int64

Geometry types:
MultiPolygon    15
Polygon         13
Name: count, dtype: int64


## Step 13 — Save the ABS Silver Dataset


The validated NSW SA4 Silver GeoDataFrame is now ready for persistent storage.

GeoParquet is used as the primary Silver format because it can preserve:

- tabular attributes;
- spatial geometry;
- coordinate reference system (CRS);
- efficient columnar storage.

After writing the file, it will be read back and validated to confirm that the spatial information survives the round trip.

In [25]:
ABS_SILVER_DIR = PROJECT_ROOT / "data" / "silver" / "abs"
ABS_SILVER_DIR.mkdir(parents=True, exist_ok=True)

ABS_SILVER_PATH = ABS_SILVER_DIR / "nsw_sa4_silver.parquet"

abs_sa4_silver.to_parquet(
    ABS_SILVER_PATH,
    index=False,
)

print("Saved to:", ABS_SILVER_PATH)
print("Exists:", ABS_SILVER_PATH.exists())
print("Size:", ABS_SILVER_PATH.stat().st_size, "bytes")

Saved to: D:\mycode\nsw-ev-data-pipeline\data\silver\abs\nsw_sa4_silver.parquet
Exists: True
Size: 11718708 bytes


## Step 14 — Validate the GeoParquet Round Trip


The Silver GeoParquet file is read back with GeoPandas and compared with the in-memory Silver GeoDataFrame.

The validation checks:

- row count;
- column schema;
- CRS preservation;
- missing values;
- geometry validity;
- geometry types;
- SA4 identifiers;
- geometry preservation.

In [27]:
abs_sa4_roundtrip = gpd.read_parquet(ABS_SILVER_PATH)

print("Type:", type(abs_sa4_roundtrip))
print("Shape:", abs_sa4_roundtrip.shape)
print("Columns:", abs_sa4_roundtrip.columns.tolist())
print("CRS:", abs_sa4_roundtrip.crs)
print("Missing values:", abs_sa4_roundtrip.isna().sum().sum())

print("\nGeometry validity:")
print(abs_sa4_roundtrip.geometry.is_valid.value_counts())

print("\nGeometry types:")
print(abs_sa4_roundtrip.geom_type.value_counts())

Type: <class 'geopandas.geodataframe.GeoDataFrame'>
Shape: (28, 8)
Columns: ['SA4_CODE26', 'SA4_NAME26', 'GCC_CODE26', 'GCC_NAME26', 'STE_CODE26', 'STE_NAME26', 'AREASQKM26', 'geometry']
CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "GDA2020", "datum": {"type": "GeodeticReferenceFrame", "name": "Geocentric Datum of Australia 2020", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope": "Geodesy, cadastre, engineering survey, topographic mapping.", "area": "Australia including Lord Howe Island, Macquarie Island, Ashmore and Cartier Islands, Christmas Island, Cocos (Keeling) Islands, Norfolk Island. All onshore and offshore.", "bbox":

In [28]:
assert abs_sa4_roundtrip.shape == abs_sa4_silver.shape

assert abs_sa4_roundtrip.columns.tolist() == abs_sa4_silver.columns.tolist()

assert abs_sa4_roundtrip.crs == abs_sa4_silver.crs

assert abs_sa4_roundtrip.isna().sum().sum() == 0

assert abs_sa4_roundtrip.geometry.is_valid.all()

assert (
    abs_sa4_roundtrip["SA4_CODE26"].tolist()
    == abs_sa4_silver["SA4_CODE26"].tolist()
)

assert (
    abs_sa4_roundtrip["SA4_NAME26"].tolist()
    == abs_sa4_silver["SA4_NAME26"].tolist()
)

assert (
    abs_sa4_roundtrip.geometry.to_wkb().tolist()
    == abs_sa4_silver.geometry.to_wkb().tolist()
)

print("ABS Silver round-trip validation passed.")

ABS Silver round-trip validation passed.
